# Trend2POC — LangGraph Workflow Graph

In [ ]:
import sys
sys.path.insert(0, '/Users/nimantha/Nimantha-Git/APP/ARAS ')


In [ ]:
from langgraph.graph import StateGraph, END, START
from IPython.display import Image, display

from backend.app.graph.state import Trend2POCState
from backend.app.graph.routing import (
    route_after_topic_approval,
    route_after_source_verification,
    route_after_report_approval,
    route_after_code_review,
    route_after_evaluation,
    route_after_github_approval,
)

def _noop(state: Trend2POCState) -> Trend2POCState:
    return state

builder = StateGraph(Trend2POCState)

for node in [
    "discovery",
    "await_topic_approval",
    "research",
    "source_verification",
    "technical_analysis",
    "report_writer",
    "await_report_approval",
    "poc_planner",
    "code_generator",
    "code_reviewer",
    "evaluator",
    "await_improvement_approval",
    "improvement",
    "await_github_approval",
    "github_publisher",
]:
    builder.add_node(node, _noop)

builder.add_edge(START, "discovery")
builder.add_edge("discovery", "await_topic_approval")

builder.add_conditional_edges(
    "await_topic_approval", route_after_topic_approval,
    {"research": "research", "end_awaiting_topic": END},
)

builder.add_edge("research", "source_verification")

builder.add_conditional_edges(
    "source_verification", route_after_source_verification,
    {"research": "research", "technical_analysis": "technical_analysis"},
)

builder.add_edge("technical_analysis", "report_writer")
builder.add_edge("report_writer", "await_report_approval")

builder.add_conditional_edges(
    "await_report_approval", route_after_report_approval,
    {"poc_planner": "poc_planner", "end_awaiting_report": END},
)

builder.add_edge("poc_planner", "code_generator")
builder.add_edge("code_generator", "code_reviewer")

builder.add_conditional_edges(
    "code_reviewer", route_after_code_review,
    {"code_generator": "code_generator", "evaluator": "evaluator"},
)

builder.add_conditional_edges(
    "evaluator", route_after_evaluation,
    {"await_github_approval": "await_github_approval", "end_eval_failed": END},
)

builder.add_edge("await_improvement_approval", "improvement")
builder.add_edge("improvement", "await_github_approval")

builder.add_conditional_edges(
    "await_github_approval", route_after_github_approval,
    {"github_publisher": "github_publisher", "end_awaiting_github": END},
)

builder.add_edge("github_publisher", END)

app = builder.compile()
print("Graph compiled")


In [ ]:
from IPython.display import Image, display

png = app.get_graph().draw_mermaid_png()

with open("workflow_graph.png", "wb") as f:
    f.write(png)

print(f"Saved workflow_graph.png ({len(png):,} bytes)")
display(Image(png))
